# Protein-Protein Osmotic Second Virial Coefficient

Calculates the potential of mean force and the second virial coefficient, $B_2$,
between two proteins using [duello](https://github.com/mlund/duello).
Atomistic PDB structures are coarse grained to one bead per amino acid with
pH-dependent charges, and interactions use the Calvados 3 force field.
The angular space is sampled on adaptive icosphere grids.


In [ ]:
# @title Install
!pip install duello --quiet
import json
import matplotlib.pyplot as plt
import numpy as np
from google.colab import files
%matplotlib inline


# Upload PDB files

Upload one PDB file for a homodimer, or two for a heterodimer.


In [ ]:
# @title Upload
uploaded = sorted(files.upload())
if not 1 <= len(uploaded) <= 2 or not all(f.endswith(".pdb") for f in uploaded):
    raise ValueError("Upload one or two PDB files.")
mol1, mol2 = uploaded[0], uploaded[-1]
print(f"mol1: {mol1}, mol2: {mol2}")


# Run

Coarse graining writes `<name>.xyz` and `<name>_topology.yaml` next to the input,
which can be reused with `duello scan --top`.


In [ ]:
# @title Input parameters { display-mode: "form" }

# @markdown #### pH:
ph = 7.0 # @param {type:"number"}
# @markdown #### Ionic strength (mol/l):
ionic_strength = 0.1 # @param {type:"number"}
# @markdown #### Temperature (kelvin):
temperature = 298.15 # @param {"type":"number"}
# @markdown #### Angular resolution (icosphere subdivisions; higher is finer and slower):
max_ndiv = 2 # @param {"type":"slider","min":0,"max":3,"step":1}
# @markdown #### Minimum mass-center separation (angstrom):
min_distance = 30 # @param {"type":"number"}
# @markdown #### Maximum mass-center separation (angstrom):
max_distance = 80 # @param {"type":"number"}
# @markdown #### Mass-center separation steps (angstrom):
distance_step = 2 # @param {"type":"number"}

!duello scan -1 "{mol1}" -2 "{mol2}" \
--ph {ph} \
--molarity {ionic_strength} \
--temperature {temperature} \
--max-ndiv {max_ndiv} \
--rmin {min_distance} --rmax {max_distance} --dr {distance_step} \
--pmf pmf.dat

virial = json.load(open("pmf.json"))
print(f"B2 = {virial['B2']:.4g} Å³, B2/B2_hs = {virial['B2_reduced']:.3g}")

r, w, u = np.loadtxt("pmf.dat", usecols=(0, 1, 2), unpack=True)
plt.plot(r, w, label="Free energy", color="orange", lw=4, alpha=0.7)
plt.plot(r, u, label="Mean energy", color="red", lw=4, alpha=0.7)
plt.legend(loc=0, frameon=False)
plt.xlabel("Mass center separation (Å)")
plt.ylabel("Energy (kT)")
